# Lanzador automatizado en Google Colab Pro (A100 / G4)

Este notebook:
1. Verifica la GPU asignada y monta Google Drive.
2. Clona o actualiza (`git pull`) el repositorio e instala dependencias (`timm`, `pyyaml`).
3. Copia/descomprime el dataset al disco local de la VM (`/content/data_local`) para evitar cuellos de botella de I/O con Drive.
4. Ejecuta los scripts `.py` por terminal guardando métricas por época y checkpoints reanudables directamente en Drive.
5. Libera automáticamente el runtime (`google.colab.runtime.unassign()`) al finalizar.

In [ ]:
# === 1. Configuración general ===
REPO_URL = "https://github.com/fhurtadoa/ProyectoDL-Grupo2-Glaucoma.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/ProyectoDL-Grupo2-Glaucoma"

# Origen del dataset en Google Drive (zip o carpeta). Si no existen, usará data/ del repo si está presente.
DRIVE_DATA_ZIP = "/content/drive/MyDrive/Glaucoma/data_glaucoma.zip"
DRIVE_DATA_DIR = "/content/drive/MyDrive/Glaucoma/data"
LOCAL_DATA_DIR = "/content/data_local"

# Salidas persistentes en Google Drive (reanudables)
OUTPUT_DIR = "/content/drive/MyDrive/Glaucoma_Colab_Results"

# Suite YAML a ejecutar (o cambia CMD_EXTRA para un comando puntual de colab.run)
SUITE_CONFIG = "colab/configs/cnn_lodo_a100.yaml"
AUTO_UNASSIGN = True  # Libera el runtime automáticamente al terminar

In [ ]:
# === 2. Verificación de GPU y montaje de Google Drive ===
!nvidia-smi
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

In [ ]:
# === 3. Clonar o actualizar repositorio e instalar dependencias ===
import os, subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

%cd {REPO_DIR}
!pip install -q timm pyyaml
!python -c "import torch, timm; print('torch:', torch.__version__, '| bf16:', torch.cuda.is_bf16_supported(), '| timm:', timm.__version__)"

In [ ]:
# === 4. Copiar/descomprimir dataset en el disco local de la VM (/content/data_local) ===
from pathlib import Path
import shutil, subprocess

local_dir = Path(LOCAL_DATA_DIR)
req_folders = ["MOONDREAM_STAGED", "Database", "RIM_ONE_DL", "EYEPACS_REAL"]

def has_cohorts(base: Path) -> bool:
    return all((base / d).is_dir() or (base / "data" / d).is_dir() for d in req_folders)

if has_cohorts(local_dir):
    print(f"[OK] Dataset ya disponible en disco local: {local_dir}")
elif Path(DRIVE_DATA_ZIP).is_file():
    local_dir.mkdir(parents=True, exist_ok=True)
    tmp_zip = "/content/_tmp_data.zip"
    print(f"Copiando {DRIVE_DATA_ZIP} -> {tmp_zip}...")
    shutil.copy2(DRIVE_DATA_ZIP, tmp_zip)
    print(f"Descomprimiendo en {local_dir}...")
    subprocess.run(["unzip", "-q", "-o", tmp_zip, "-d", str(local_dir)], check=True)
    Path(tmp_zip).unlink(missing_ok=True)
elif Path(DRIVE_DATA_DIR).is_dir():
    print(f"Copiando carpeta {DRIVE_DATA_DIR} -> {local_dir}...")
    shutil.copytree(DRIVE_DATA_DIR, local_dir, dirs_exist_ok=True)
elif has_cohorts(Path(REPO_DIR)):
    print(f"Copiando data/ del repo -> {local_dir}...")
    shutil.copytree(Path(REPO_DIR) / "data", local_dir, dirs_exist_ok=True)
else:
    raise FileNotFoundError(
        f"No se encontró el dataset en {DRIVE_DATA_ZIP}, {DRIVE_DATA_DIR} ni {REPO_DIR}/data."
    )

assert has_cohorts(local_dir), f"Faltan cohortes requeridas en {local_dir}: {req_folders}"
print("[VERIFICADO] Las 4 cohortes están listas en", local_dir)

In [ ]:
# === 5. Ejecución por terminal (reanudable) y liberación automática del runtime ===
import os, subprocess, sys
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, "-u", "-m", "colab.run_suite",
    "--config", SUITE_CONFIG,
    "--data_dir", LOCAL_DATA_DIR,
    "--output_dir", OUTPUT_DIR,
]
print("Ejecutando:", " ".join(cmd))

try:
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"La ejecución terminó con código {ret}")
finally:
    # Fuerza sincronización del sistema de archivos antes de desconectar
    os.sync()
    if AUTO_UNASSIGN:
        print("\n[COLAB] Liberando runtime automáticamente para ahorrar Compute Units...")
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception as e:
            print("No se pudo liberar runtime automáticamente:", e)

In [ ]:
# === 6. (Opcional, ejecutar con AUTO_UNASSIGN=False) Inspección rápida de resultados ===
import pandas as pd
from pathlib import Path

out = Path(OUTPUT_DIR)
for name in ["suite_summary.csv", "exp05_lodo/exp05_lodo_summary.csv", "exp04/exp04_summary.csv", "horizon/horizon_summary.csv"]:
    f = out / name
    if f.is_file():
        print(f"\n=== {name} ===")
        display(pd.read_csv(f))